In [ ]:
!pip -q install pandas numpy openpyxl factor_analyzer semopy

import re
import numpy as np
import pandas as pd
from google.colab import files

from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity
from semopy import Model
from semopy.stats import calc_stats

# ---------- Upload ----------
uploaded = files.upload()
excel_file = next(iter(uploaded.keys()))
print(f"Uploaded file: {excel_file}")

# ---------- Config ----------
SHEET_NAME = "Metrics"
ROTATION = "promax"
EFA_LOADING_THRESHOLD = 0.40
CFA_LOADING_THRESHOLD = 0.50
ROUND_DIGITS = 4


# 1) Reliability / validity metrics

def cronbach_alpha(df: pd.DataFrame) -> float:
    X = df.apply(pd.to_numeric, errors="coerce")
    X = X.dropna(axis=0, how="all")

    if X.shape[0] < 2 or X.shape[1] < 2:
        return np.nan

    X = X.fillna(X.mean(numeric_only=True))

    k = X.shape[1]
    item_var = X.var(axis=0, ddof=1)
    total_var = X.sum(axis=1).var(ddof=1)

    if total_var <= 0 or np.isnan(total_var):
        return np.nan

    return float((k / (k - 1)) * (1 - item_var.sum() / total_var))


def composite_reliability(loadings: np.ndarray) -> float:
    lam = np.asarray(loadings, dtype=float)
    lam = lam[~np.isnan(lam)]

    if len(lam) < 2:
        return np.nan

    numerator = np.sum(lam) ** 2
    denominator = numerator + np.sum(1 - lam**2)

    if denominator == 0:
        return np.nan

    return float(numerator / denominator)


def average_variance_extracted(loadings: np.ndarray) -> float:
    lam = np.asarray(loadings, dtype=float)
    lam = lam[~np.isnan(lam)]

    if len(lam) < 1:
        return np.nan

    return float(np.sum(lam**2) / len(lam))


def interpret_alpha(alpha: float) -> str:
    if pd.isna(alpha):
        return "NA"
    if alpha >= 0.90:
        return "Excellent"
    if alpha >= 0.80:
        return "Good"
    if alpha >= 0.70:
        return "Acceptable"
    if alpha >= 0.60:
        return "Questionable"
    return "Weak"


def interpret_cr(cr: float) -> str:
    if pd.isna(cr):
        return "NA"
    return "Acceptable" if cr >= 0.70 else "Low"


def interpret_ave(ave: float) -> str:
    if pd.isna(ave):
        return "NA"
    return "Convergent validity supported" if ave >= 0.50 else "Below threshold"

# 2) Parse Metrics sheet

def _is_block_header(cell) -> bool:
    if not isinstance(cell, str):
        return False
    s = cell.strip()
    return (s == "Critical Success Factors") or s.startswith("KPI -")


def _is_respondent_row(cell) -> bool:
    return isinstance(cell, str) and cell.strip().startswith("Respondent")


def extract_blocks_from_metrics(metrics_raw: pd.DataFrame):
    df = metrics_raw.copy()
    first_col = df.iloc[:, 0]
    header_idxs = [i for i, v in enumerate(first_col.values) if _is_block_header(v)]

    for idx_pos, hdr_i in enumerate(header_idxs):
        block_name = str(df.iat[hdr_i, 0]).strip()

        header_row = df.iloc[hdr_i, :]
        item_cols = []
        for j in range(1, df.shape[1]):
            val = header_row.iat[j]
            if pd.isna(val) or (isinstance(val, str) and val.strip() == ""):
                break
            item_cols.append(j)

        if len(item_cols) < 2:
            continue

        start = hdr_i + 1
        end = header_idxs[idx_pos + 1] if idx_pos + 1 < len(header_idxs) else df.shape[0]
        sub = df.iloc[start:end, :]

        sub = sub[sub.iloc[:, 0].apply(_is_respondent_row)]
        if sub.empty:
            continue

        X = sub.iloc[:, item_cols].copy()
        X.columns = [str(header_row.iat[j]).strip() for j in item_cols]
        X.index = sub.iloc[:, 0].astype(str).str.strip()

        X = X.apply(pd.to_numeric, errors="coerce")
        X = X.dropna(axis=1, how="all")
        X = X.dropna(axis=0, how="all")

        if X.shape[0] >= 3 and X.shape[1] >= 2:
            yield block_name, X

# 3) Safe-name utilities for semopy

def make_safe_name(name: str) -> str:
    """
    Convert labels like 'CSF 1' -> 'CSF_1'
    Keeps only letters, digits, and underscores.
    """
    safe = re.sub(r"\W+", "_", str(name).strip())
    safe = re.sub(r"_+", "_", safe).strip("_")

    if not safe:
        safe = "item"

    if safe[0].isdigit():
        safe = f"v_{safe}"

    return safe


def make_safe_columns(df: pd.DataFrame):
    """
    Returns:
    - df_safe: dataframe with semopy-safe columns
    - orig_to_safe: original -> safe
    - safe_to_orig: safe -> original
    """
    orig_to_safe = {}
    safe_to_orig = {}
    used = set()

    for col in df.columns:
        base = make_safe_name(col)
        safe = base
        i = 1
        while safe in used:
            safe = f"{base}_{i}"
            i += 1

        used.add(safe)
        orig_to_safe[col] = safe
        safe_to_orig[safe] = col

    df_safe = df.rename(columns=orig_to_safe).copy()
    return df_safe, orig_to_safe, safe_to_orig

# 4) EFA

def choose_n_factors_eigenvalue_rule(X: pd.DataFrame, max_factors: int = None) -> int:
    if max_factors is None:
        max_factors = min(X.shape[1], X.shape[0] - 1)

    fa0 = FactorAnalyzer(rotation=None, n_factors=min(X.shape[1], max_factors))
    fa0.fit(X.fillna(X.mean(numeric_only=True)))

    ev, _ = fa0.get_eigenvalues()
    n = int(np.sum(ev > 1.0))
    n = max(1, min(n, X.shape[1]))
    return n


def run_efa(X: pd.DataFrame, n_factors: int = None, rotation: str = ROTATION):
    X_imp = X.fillna(X.mean(numeric_only=True))

    kmo_per_item, kmo_model = calculate_kmo(X_imp)
    bart_chi2, bart_p = calculate_bartlett_sphericity(X_imp)

    if n_factors is None:
        n_factors = choose_n_factors_eigenvalue_rule(X_imp)

    fa = FactorAnalyzer(n_factors=n_factors, rotation=rotation, method="minres")
    fa.fit(X_imp)

    loading_cols = [f"Factor_{i+1}" for i in range(n_factors)]
    loadings_df = pd.DataFrame(fa.loadings_, index=X.columns, columns=loading_cols)

    return {
        "kmo": float(kmo_model),
        "bartlett_chi2": float(bart_chi2),
        "bartlett_p": float(bart_p),
        "n_factors": int(n_factors),
        "efa_loadings": loadings_df
    }


def assign_items_to_factors(efa_loadings: pd.DataFrame, threshold: float = EFA_LOADING_THRESHOLD):
    assignments = {}
    for item, row in efa_loadings.iterrows():
        abs_row = row.abs()
        best_factor = abs_row.idxmax()
        best_loading = abs_row.max()

        if best_loading >= threshold:
            assignments.setdefault(best_factor, []).append(item)

    return assignments

def merge_to_single_factor(assignments):
    all_items = []
    for items in assignments.values():
        all_items.extend(items)

    all_items = list(dict.fromkeys(all_items))
    return {"Factor_1": all_items}

# 5) CFA with safe names

def build_cfa_model_syntax(assignments_safe: dict) -> str:
    lines = []
    for factor, items in assignments_safe.items():
        if len(items) >= 2:
            lines.append(f"{factor} =~ " + " + ".join(items))
    return "\n".join(lines)


def run_cfa(X_original: pd.DataFrame, assignments_original: dict):
    """
    Convert item names to safe semopy names, fit CFA, and map results back.
    """
    X_safe, orig_to_safe, safe_to_orig = make_safe_columns(X_original.fillna(X_original.mean(numeric_only=True)))

    # Convert assignments from original labels to safe labels
    assignments_safe = {
        factor: [orig_to_safe[item] for item in items if item in orig_to_safe]
        for factor, items in assignments_original.items()
    }

    # Keep only factors with >= 2 items
    assignments_safe = {k: v for k, v in assignments_safe.items() if len(v) >= 2}

    if not assignments_safe:
        raise ValueError("No valid factor assignments remained after safe-name conversion.")

    model_desc = build_cfa_model_syntax(assignments_safe)
    print("\nCFA model syntax:\n", model_desc)

    model = Model(model_desc)
    model.fit(X_safe)

    estimates = model.inspect(std_est=True)
    stats = calc_stats(model)

    std_col = "Est. Std" if "Est. Std" in estimates.columns else "Std. Ests"

    # Measurement loadings in semopy appear as: item ~ factor
    loadings = estimates[
        (estimates["op"] == "~") &
        (estimates["rval"].str.startswith("Factor_"))
    ].copy()

    loadings = loadings.rename(columns={std_col: "std_loading"})
    loadings = loadings[["lval", "rval", "Estimate", "std_loading"]]
    loadings.columns = ["item_safe", "factor", "estimate", "std_loading"]

    # Map back to original item labels
    loadings["item"] = loadings["item_safe"].map(safe_to_orig)
    loadings = loadings[["item", "factor", "estimate", "std_loading"]]

    return {
        "model": model,
        "fit_stats": stats,
        "std_loadings": loadings,
        "model_desc": model_desc
    }

# 6) Per-block analysis

def analyze_block(block_name: str, X: pd.DataFrame):
    alpha_block = cronbach_alpha(X)

    efa = run_efa(X)
    efa_loadings = efa["efa_loadings"]
    assignments = assign_items_to_factors(efa_loadings)

    if block_name.strip().lower() == "kpi - org & training":
      assignments = merge_to_single_factor(assignments)

    assignments = {k: v for k, v in assignments.items() if len(v) >= 2}

    if len(assignments) == 0:
        return {
            "block_summary": pd.DataFrame([{
                "block": block_name,
                "n_respondents": X.shape[0],
                "n_items": X.shape[1],
                "CA": round(alpha_block, ROUND_DIGITS) if pd.notna(alpha_block) else np.nan,
                "CA_interpretation": interpret_alpha(alpha_block),
                "KMO": round(efa["kmo"], ROUND_DIGITS),
                "Bartlett_p": round(efa["bartlett_p"], ROUND_DIGITS),
                "EFA_n_factors": efa["n_factors"],
                "CFA_status": "Skipped - no usable factor assignment",
            }]),
            "efa_loadings": efa_loadings.round(ROUND_DIGITS),
            "cfa_loadings": pd.DataFrame(),
            "factor_metrics": pd.DataFrame(),
            "fit_stats": pd.DataFrame()
        }

    try:
        cfa = run_cfa(X, assignments)
        cfa_loadings = cfa["std_loadings"].copy()
        cfa_status = "Success"
    except Exception as e:
        return {
            "block_summary": pd.DataFrame([{
                "block": block_name,
                "n_respondents": X.shape[0],
                "n_items": X.shape[1],
                "CA": round(alpha_block, ROUND_DIGITS) if pd.notna(alpha_block) else np.nan,
                "CA_interpretation": interpret_alpha(alpha_block),
                "KMO": round(efa["kmo"], ROUND_DIGITS),
                "Bartlett_p": round(efa["bartlett_p"], ROUND_DIGITS),
                "EFA_n_factors": efa["n_factors"],
                "CFA_status": f"Failed - {str(e)}",
            }]),
            "efa_loadings": efa_loadings.round(ROUND_DIGITS),
            "cfa_loadings": pd.DataFrame(),
            "factor_metrics": pd.DataFrame(),
            "fit_stats": pd.DataFrame()
        }

    factor_rows = []
    for factor, items in assignments.items():
        sub = X[items].copy()
        ca = cronbach_alpha(sub)

        sub_loadings = cfa_loadings[cfa_loadings["factor"] == factor].copy()
        sub_loadings["abs_std_loading"] = sub_loadings["std_loading"].abs()

        valid_loadings = sub_loadings.loc[
            sub_loadings["abs_std_loading"] >= CFA_LOADING_THRESHOLD,
            "abs_std_loading"
        ].values

        if len(valid_loadings) < 2:
            cr = np.nan
            ave = np.nan
        else:
            cr = composite_reliability(valid_loadings)
            ave = average_variance_extracted(valid_loadings)

        factor_rows.append({
            "block": block_name,
            "factor": factor,
            "n_items": len(items),
            "items": ", ".join(items),
            "CA": round(ca, ROUND_DIGITS) if pd.notna(ca) else np.nan,
            "CA_interpretation": interpret_alpha(ca),
            "CR": round(cr, ROUND_DIGITS) if pd.notna(cr) else np.nan,
            "CR_interpretation": interpret_cr(cr),
            "AVE": round(ave, ROUND_DIGITS) if pd.notna(ave) else np.nan,
            "AVE_interpretation": interpret_ave(ave)
        })

    factor_metrics = pd.DataFrame(factor_rows)

    fit_stats = cfa["fit_stats"].copy()
    fit_stats.index = [block_name]

    block_summary = pd.DataFrame([{
        "block": block_name,
        "n_respondents": X.shape[0],
        "n_items": X.shape[1],
        "CA": round(alpha_block, ROUND_DIGITS) if pd.notna(alpha_block) else np.nan,
        "CA_interpretation": interpret_alpha(alpha_block),
        "KMO": round(efa["kmo"], ROUND_DIGITS),
        "Bartlett_p": round(efa["bartlett_p"], ROUND_DIGITS),
        "EFA_n_factors": efa["n_factors"],
        "CFA_status": cfa_status
    }])

    return {
        "block_summary": block_summary,
        "efa_loadings": efa_loadings.round(ROUND_DIGITS),
        "cfa_loadings": cfa_loadings.round(ROUND_DIGITS),
        "factor_metrics": factor_metrics,
        "fit_stats": fit_stats.round(ROUND_DIGITS)
    }

# 7) Master runner

def run_full_pipeline(excel_path: str, sheet_name: str = SHEET_NAME):
    raw = pd.read_excel(excel_path, sheet_name=sheet_name, header=None)

    all_block_summaries = []
    all_factor_metrics = []
    all_fit_stats = []
    efa_tables = {}
    cfa_tables = {}

    for block_name, X in extract_blocks_from_metrics(raw):
        result = analyze_block(block_name, X)

        all_block_summaries.append(result["block_summary"])
        if not result["factor_metrics"].empty:
            all_factor_metrics.append(result["factor_metrics"])
        if not result["fit_stats"].empty:
            all_fit_stats.append(result["fit_stats"])

        efa_tables[block_name] = result["efa_loadings"]
        cfa_tables[block_name] = result["cfa_loadings"]

    block_summary_df = pd.concat(all_block_summaries, ignore_index=True) if all_block_summaries else pd.DataFrame()
    factor_metrics_df = pd.concat(all_factor_metrics, ignore_index=True) if all_factor_metrics else pd.DataFrame()
    fit_stats_df = pd.concat(all_fit_stats, axis=0) if all_fit_stats else pd.DataFrame()

    return {
        "block_summary": block_summary_df,
        "factor_metrics": factor_metrics_df,
        "fit_stats": fit_stats_df,
        "efa_tables": efa_tables,
        "cfa_tables": cfa_tables
    }

# 8) Run

results = run_full_pipeline(excel_file, sheet_name=SHEET_NAME)

print("\n================ BLOCK SUMMARY ================\n")
print(results["block_summary"].to_string(index=False) if not results["block_summary"].empty else "No blocks found.")

print("\n================ FACTOR METRICS (CA / CR / AVE) ================\n")
print(results["factor_metrics"].to_string(index=False) if not results["factor_metrics"].empty else "No factor metrics generated.")

print("\n================ CFA FIT STATS ================\n")
print(results["fit_stats"].to_string() if not results["fit_stats"].empty else "No fit statistics generated.")

for block_name, table in results["efa_tables"].items():
    print(f"\n================ EFA LOADINGS: {block_name} ================\n")
    print(table.to_string())

for block_name, table in results["cfa_tables"].items():
    print(f"\n================ CFA STANDARDIZED LOADINGS: {block_name} ================\n")
    print(table.to_string(index=False) if not table.empty else "No CFA loadings.")

# Save outputs
results["block_summary"].to_csv("block_summary.csv", index=False)
results["factor_metrics"].to_csv("factor_metrics_CA_CR_AVE.csv", index=False)
results["fit_stats"].to_csv("cfa_fit_stats.csv")

with pd.ExcelWriter("measurement_validation_output.xlsx", engine="openpyxl") as writer:
    results["block_summary"].to_excel(writer, sheet_name="Block_Summary", index=False)
    results["factor_metrics"].to_excel(writer, sheet_name="Factor_Metrics", index=False)
    results["fit_stats"].to_excel(writer, sheet_name="CFA_Fit_Stats")

    for block_name, table in results["efa_tables"].items():
        safe_name = ("EFA_" + block_name)[:31]
        table.to_excel(writer, sheet_name=safe_name)

    for block_name, table in results["cfa_tables"].items():
        safe_name = ("CFA_" + block_name)[:31]
        table.to_excel(writer, sheet_name=safe_name, index=False)

print("\nSaved files:")
print("- block_summary.csv")
print("- factor_metrics_CA_CR_AVE.csv")
print("- cfa_fit_stats.csv")
print("- measurement_validation_output.xlsx")

files.download("measurement_validation_output.xlsx")

Saving Active Shooter Survey_metadata (Quantitative Results).xlsx to Active Shooter Survey_metadata (Quantitative Results) (2).xlsx
Uploaded file: Active Shooter Survey_metadata (Quantitative Results) (2).xlsx

CFA model syntax:
 Factor_1 =~ CSF_1 + CSF_2 + CSF_3 + CSF_4 + CSF_5

CFA model syntax:
 Factor_1 =~ KPI_1 + KPI_2 + KPI_3
Factor_2 =~ KPI_4 + KPI_5


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/factor_analyzer/factor_analyzer.py:663: UserWarning: No rotation will be performed when the number of factors equals 1.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skl


CFA model syntax:
 Factor_1 =~ KPI_1 + KPI_2 + KPI_3 + KPI_5

CFA model syntax:
 Factor_1 =~ KPI_1 + KPI_2 + KPI_3 + KPI_5

CFA model syntax:
 Factor_1 =~ KPI_1 + KPI_4 + KPI_2 + KPI_3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/factor_analyzer/factor_analyzer.py:663: UserWarning: No rotation will be performed when the number of factors equals 1.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/skl


CFA model syntax:
 Factor_1 =~ KPI_1 + KPI_2 + KPI_3

================ BLOCK SUMMARY ================

                               block  n_respondents  n_items     CA CA_interpretation    KMO  Bartlett_p  EFA_n_factors CFA_status
            Critical Success Factors             30        5 0.7989        Acceptable 0.7145      0.0000              1    Success
KPI - Building Layout & Arch Factors             30        5 0.7900        Acceptable 0.6625      0.0000              2    Success
           KPI - Respondent Behavior             30        5 0.7467        Acceptable 0.6161      0.0001              1    Success
                  KPI - Tech & Comms             30        5 0.7990        Acceptable 0.7709      0.0000              1    Success
                KPI - Org & Training             30        4 0.7022        Acceptable 0.6074      0.0002              2    Success
           KPI - Enviro & Contextual             30        3 0.6932      Questionable 0.6545      0.0017      

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>